In [ ]:
from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, t as student_t
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset


DATA_DIR = Path("data")
BASE_OUTPUT_DIR = Path("results") / "graphlet_abg_corrected_hpb_repeated_5seeds"

SEED = 42
ALL_SEEDS = [SEED, 52, 62, 72, 82]
TRAIN_SEEDS = ALL_SEEDS.copy()
SKIP_COMPLETED = True

TARGET_FILE = "processed_combination_response_r070_clean_with_qc.csv"
DRUG_FEATURE_FILE = "Graphlet_features_6_standardized.csv"
CELL_FEATURE_FILE = "cell_features_977d.csv"

MAX_EPOCHS = 100
BATCH_SIZE = 8
INITIAL_LR = 5e-5
ALPHA = 0.5

LR_FACTOR = 0.5
LR_PATIENCE = 5
MIN_LR = 1e-7

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 1e-6

TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.10
TEST_FRACTION = 0.20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using device: {device}")
print(f"All reporting seeds: {ALL_SEEDS}")
print(f"Seeds to train in this notebook: {TRAIN_SEEDS}")
print(f"Output directory: {BASE_OUTPUT_DIR}")


In [ ]:
def _require_file(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path.resolve()}\n"
            "Put all three CSV files in DATA_DIR, or change DATA_DIR above."
        )


def load_data(data_dir=DATA_DIR):
    target_path = data_dir / TARGET_FILE
    drug_path = data_dir / DRUG_FEATURE_FILE
    cell_path = data_dir / CELL_FEATURE_FILE

    for path in (target_path, drug_path, cell_path):
        _require_file(path)

    combo_df = pd.read_csv(target_path, encoding="utf-8-sig")
    combo_df = combo_df.rename(columns={
        "target": "X/X0",
        "drugA_conc": "drugA Conc (µM)",
        "drugB_conc": "drugB Conc (µM)",
    })

    combo_df["drugA_name"] = combo_df["drugA_name"].astype(str).str.strip().str.upper()
    combo_df["drugB_name"] = combo_df["drugB_name"].astype(str).str.strip().str.upper()
    combo_df["cell_line"] = combo_df["cell_line"].astype(str).str.strip()

    drug_features = pd.read_csv(drug_path, encoding="utf-8-sig")
    drug_features["name"] = drug_features["name"].astype(str).str.strip().str.upper()
    drug_features = drug_features.drop_duplicates("name", keep="first").set_index("name")

    drug_features = drug_features.iloc[:, 2:].apply(pd.to_numeric, errors="coerce")
    drug_array = drug_features.to_numpy(dtype=np.float32)
    if not np.isfinite(drug_array).all():
        bad_count = int((~np.isfinite(drug_array)).sum())
        raise ValueError(f"Drug feature table contains {bad_count} NaN/Inf values")
    drug_features = drug_features.astype(np.float32)

    cell_features = pd.read_csv(cell_path, encoding="utf-8-sig")
    cell_features["Cell_Line"] = cell_features["Cell_Line"].astype(str).str.strip()
    cell_features = cell_features.drop_duplicates("Cell_Line", keep="first").set_index("Cell_Line")
    cell_features = cell_features.apply(pd.to_numeric, errors="coerce")
    cell_array = cell_features.to_numpy(dtype=np.float32)
    if not np.isfinite(cell_array).all():
        bad_count = int((~np.isfinite(cell_array)).sum())
        raise ValueError(f"Cell feature table contains {bad_count} NaN/Inf values")
    cell_features = cell_features.astype(np.float32)

    return combo_df, drug_features, cell_features


def preprocess_data(combo_df, drug_features, cell_features):
    required = [
        "drugA_name", "drugB_name", "cell_line",
        "drugA Conc (µM)", "drugB Conc (µM)",
        "single_resp_1", "single_resp_2", "X/X0",
    ]
    missing = [column for column in required if column not in combo_df.columns]
    if missing:
        raise KeyError(f"Combination response CSV is missing columns: {missing}")

    valid_mask = (
        combo_df["drugA_name"].isin(drug_features.index)
        & combo_df["drugB_name"].isin(drug_features.index)
        & combo_df["cell_line"].isin(cell_features.index)
    )

    skipped_missing_features = int((~valid_mask).sum())
    data_df = combo_df.loc[valid_mask, required].copy()
    data_df = data_df.rename(columns={
        "drugA Conc (µM)": "drugA_conc",
        "drugB Conc (µM)": "drugB_conc",
        "X/X0": "target",
    })

    numeric_columns = [
        "drugA_conc", "drugB_conc", "single_resp_1", "single_resp_2", "target"
    ]
    for column in numeric_columns:
        data_df[column] = pd.to_numeric(data_df[column], errors="coerce")

    data_df[numeric_columns] = data_df[numeric_columns].replace(
        [np.inf, -np.inf], np.nan
    )

    before_numeric_drop = len(data_df)
    data_df = data_df.dropna(subset=numeric_columns).reset_index(drop=True)
    skipped_invalid_numeric = before_numeric_drop - len(data_df)


    pair_array = np.sort(data_df[["drugA_name", "drugB_name"]].to_numpy(dtype=str), axis=1)
    data_df["group_id"] = (
        pair_array[:, 0]
        + "||"
        + pair_array[:, 1]
        + "||"
        + data_df["cell_line"].astype(str).to_numpy()
    )

    print(f"Skipped {skipped_missing_features} rows due to missing drug/cell features")
    print(f"Skipped {skipped_invalid_numeric} rows due to invalid numeric values")
    print(f"Valid samples: {len(data_df)}")
    print(f"Drug feature dimension: {drug_features.shape[1]}")
    print(f"Cell feature dimension: {cell_features.shape[1]}")
    print(f"Independent drug-pair/cell-line groups: {data_df['group_id'].nunique()}")

    if len(data_df) == 0:
        raise ValueError("No valid samples remain after preprocessing.")

    return data_df


class DrugCombinationDataset(Dataset):
    def __init__(self, data_df, drug_features, cell_features):
        self.data = data_df.reset_index(drop=True)

        self.drug_names = drug_features.index.astype(str).tolist()
        self.cell_names = cell_features.index.astype(str).tolist()
        self.drug_to_index = {name: i for i, name in enumerate(self.drug_names)}
        self.cell_to_index = {name: i for i, name in enumerate(self.cell_names)}

        self.drug_matrix = np.ascontiguousarray(drug_features.to_numpy(dtype=np.float32))
        self.cell_matrix = np.ascontiguousarray(cell_features.to_numpy(dtype=np.float32))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        drug_a = row["drugA_name"]
        drug_b = row["drugB_name"]
        cell_line = row["cell_line"]

        return {
            "drugA_feat": torch.from_numpy(self.drug_matrix[self.drug_to_index[drug_a]]),
            "drugB_feat": torch.from_numpy(self.drug_matrix[self.drug_to_index[drug_b]]),
            "cell_feat": torch.from_numpy(self.cell_matrix[self.cell_to_index[cell_line]]),
            "target": torch.tensor(row["target"], dtype=torch.float32),
            "drugA_conc": torch.tensor(row["drugA_conc"], dtype=torch.float32),
            "drugB_conc": torch.tensor(row["drugB_conc"], dtype=torch.float32),
            "single_resp_1": torch.tensor(row["single_resp_1"], dtype=torch.float32),
            "single_resp_2": torch.tensor(row["single_resp_2"], dtype=torch.float32),
            "index": idx,
            "drugA_name": drug_a,
            "drugB_name": drug_b,
            "cell_line": cell_line,
        }


def grouped_train_val_test_split(data_df, seed):
    outer = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=seed)
    train_val_idx, test_idx = next(
        outer.split(data_df, groups=data_df["group_id"])
    )
    train_val_df = data_df.iloc[train_val_idx].reset_index(drop=True)
    test_df = data_df.iloc[test_idx].reset_index(drop=True)

    adjusted_val_fraction = VAL_FRACTION / (1.0 - TEST_FRACTION)
    inner = GroupShuffleSplit(n_splits=1, test_size=adjusted_val_fraction, random_state=seed + 1)
    train_idx, val_idx = next(
        inner.split(train_val_df, groups=train_val_df["group_id"])
    )
    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

    train_groups = set(train_df["group_id"])
    val_groups = set(val_df["group_id"])
    test_groups = set(test_df["group_id"])
    assert train_groups.isdisjoint(val_groups)
    assert train_groups.isdisjoint(test_groups)
    assert val_groups.isdisjoint(test_groups)

    print(f"Train samples/groups: {len(train_df)} / {len(train_groups)}")
    print(f"Validation samples/groups: {len(val_df)} / {len(val_groups)}")
    print(f"Test samples/groups: {len(test_df)} / {len(test_groups)}")
    return train_df, val_df, test_df


In [ ]:
class DrugCombinationModel(nn.Module):
    def __init__(self, drug_feat_dim=1000, cell_feat_dim=977):
        super(DrugCombinationModel, self).__init__()

        self.drug_encoder = nn.Sequential(
            nn.Linear(drug_feat_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )

        self.cell_encoder = nn.Sequential(
            nn.Linear(cell_feat_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )


        self.theta1_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

        self.theta2_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )


        self.epsilon_net = nn.Sequential(
            nn.Linear(256 * 3 + 256 + 2, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )


        self.dose_encoder = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
        )


        hpb_input_dim = 256 + 256 + 256 + 32 + 32
        self.predictor_direct = nn.Sequential(
            nn.Linear(hpb_input_dim, 2048),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        drugA_feat,
        drugB_feat,
        cell_feat,
        combo_feat,
        drugA_conc,
        drugB_conc,
    ):
        encoded_drugA = self.drug_encoder(drugA_feat)
        encoded_drugB = self.drug_encoder(drugB_feat)
        encoded_cell = self.cell_encoder(cell_feat)

        theta1_input = torch.cat(
            [encoded_drugA, encoded_cell, drugA_conc.unsqueeze(1)], dim=1
        )
        theta2_input = torch.cat(
            [encoded_drugB, encoded_cell, drugB_conc.unsqueeze(1)], dim=1
        )
        theta1_raw = self.theta1_net(theta1_input)
        theta2_raw = self.theta2_net(theta2_input)

        drug_pair_sum = encoded_drugA + encoded_drugB
        drug_pair_product = encoded_drugA * encoded_drugB
        drug_pair_abs_difference = torch.abs(encoded_drugA - encoded_drugB)
        epsilon_input = torch.cat(
            [
                drug_pair_sum,
                drug_pair_product,
                drug_pair_abs_difference,
                encoded_cell,
                drugA_conc.unsqueeze(1),
                drugB_conc.unsqueeze(1),
            ],
            dim=1,
        )
        epsilon = self.epsilon_net(epsilon_input)

        theta12_sum = theta1_raw + theta2_raw
        theta1 = theta1_raw / theta12_sum
        theta2 = theta2_raw / theta12_sum

        encoded_doseA = self.dose_encoder(drugA_conc.unsqueeze(1))
        encoded_doseB = self.dose_encoder(drugB_conc.unsqueeze(1))

        combined_features = torch.cat(
            [
                encoded_drugA,
                encoded_drugB,
                encoded_cell,
                encoded_doseA,
                encoded_doseB,
            ],
            dim=1,
        )
        p_direct = self.predictor_direct(combined_features)

        return (
            theta1.squeeze(-1),
            theta2.squeeze(-1),
            epsilon.squeeze(-1),
            p_direct.squeeze(-1),
        )


def forward_batch(model, batch, alpha=ALPHA):
    drugA_feat = batch["drugA_feat"].to(device, non_blocking=True)
    drugB_feat = batch["drugB_feat"].to(device, non_blocking=True)
    cell_feat = batch["cell_feat"].to(device, non_blocking=True)
    drugA_conc = batch["drugA_conc"].to(device, non_blocking=True)
    drugB_conc = batch["drugB_conc"].to(device, non_blocking=True)
    single_resp_1 = batch["single_resp_1"].to(device, non_blocking=True)
    single_resp_2 = batch["single_resp_2"].to(device, non_blocking=True)
    target = batch["target"].to(device, non_blocking=True)

    theta1, theta2, epsilon, p_direct = model(
        drugA_feat, drugB_feat, cell_feat, None, drugA_conc, drugB_conc
    )
    p_idb = theta1 * single_resp_1 + theta2 * single_resp_2 + epsilon
    p_final = alpha * p_idb + (1.0 - alpha) * p_direct
    return target, p_final, p_idb, p_direct, theta1, theta2, epsilon


In [ ]:
def concordance_index_continuous(targets, predictions):

    targets = np.asarray(targets, dtype=np.float64)
    predictions = np.asarray(predictions, dtype=np.float64)
    mask = np.isfinite(targets) & np.isfinite(predictions)
    targets = targets[mask]
    predictions = predictions[mask]

    if len(targets) < 2:
        return np.nan

    order = np.argsort(targets, kind="mergesort")
    targets = targets[order]
    predictions = predictions[order]
    _, pred_ranks = np.unique(predictions, return_inverse=True)
    pred_ranks = pred_ranks + 1
    tree = np.zeros(int(pred_ranks.max()) + 1, dtype=np.int64)

    def update(index):
        while index < len(tree):
            tree[index] += 1
            index += index & -index

    def query(index):
        count = 0
        while index > 0:
            count += tree[index]
            index -= index & -index
        return count

    concordant = 0.0
    comparable = 0
    previous_count = 0
    start = 0
    while start < len(targets):
        end = start + 1
        while end < len(targets) and targets[end] == targets[start]:
            end += 1

        for rank in pred_ranks[start:end]:
            less = query(int(rank) - 1)
            less_or_equal = query(int(rank))
            equal = less_or_equal - less
            concordant += less + 0.5 * equal
            comparable += previous_count

        for rank in pred_ranks[start:end]:
            update(int(rank))
        previous_count += end - start
        start = end

    return concordant / comparable if comparable else np.nan


def regression_metrics(targets, predictions, include_c_index=False):
    targets = np.asarray(targets, dtype=np.float64)
    predictions = np.asarray(predictions, dtype=np.float64)
    if not np.isfinite(targets).all() or not np.isfinite(predictions).all():
        raise ValueError("Targets or predictions contain NaN/Inf values")

    mse = mean_squared_error(targets, predictions)
    metrics = {
        "mse": mse,
        "rmse": np.sqrt(mse),
        "mae": mean_absolute_error(targets, predictions),
        "pcc": pearsonr(targets, predictions)[0]
        if np.std(targets) > 0 and np.std(predictions) > 0
        else np.nan,
        "r2": r2_score(targets, predictions),
    }
    if include_c_index:
        metrics["c_index"] = concordance_index_continuous(targets, predictions)
    return metrics


@torch.no_grad()
def evaluate_model(model, data_loader, include_c_index=False):
    model.eval()
    total_loss = 0.0
    all_targets = []
    all_predictions = []
    criterion = nn.MSELoss()

    for batch in data_loader:
        target, p_final, *_ = forward_batch(model, batch)
        if not torch.isfinite(p_final).all():
            raise FloatingPointError("Non-finite prediction detected")
        loss = criterion(p_final, target)
        total_loss += loss.item() * target.size(0)
        all_targets.extend(target.detach().cpu().numpy())
        all_predictions.extend(p_final.detach().cpu().numpy())

    mean_loss = total_loss / len(data_loader.dataset)
    metrics = regression_metrics(
        all_targets, all_predictions, include_c_index=include_c_index
    )
    return mean_loss, metrics


def train_model(model, train_loader, val_loader, run_seed, output_dir):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=INITIAL_LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        min_lr=MIN_LR,
    )

    model = model.to(device)
    best_val_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    history = []
    checkpoint_path = output_dir / "best_model_group_split_drug_fusion.pt"

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            optimizer.zero_grad()
            target, p_final, *_ = forward_batch(model, batch)
            loss = criterion(p_final, target)
            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss at seed {run_seed}, epoch {epoch}"
                )
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * target.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_loss, val_metrics = evaluate_model(model, val_loader)
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        improved = val_loss < best_val_loss - MIN_DELTA
        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "best_val_loss": best_val_loss,
                    "seed": run_seed,
                    "alpha": ALPHA,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1

        history.append({
            "seed": run_seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_rmse": val_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "val_pcc": val_metrics["pcc"],
            "val_r2": val_metrics["r2"],
            "learning_rate": current_lr,
            "is_best": improved,
        })

        print(
            f"Seed {run_seed} | Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"train={train_loss:.6f} | val={val_loss:.6f} | "
            f"RMSE={val_metrics['rmse']:.6f} | "
            f"PCC={val_metrics['pcc']:.6f} | R2={val_metrics['r2']:.6f} | "
            f"LR={current_lr:.2e}"
        )

        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"Seed {run_seed}: early stopping at epoch {epoch}; "
                f"best epoch={best_epoch}, best validation MSE={best_val_loss:.6f}"
            )
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(output_dir / "training_history.csv", index=False)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model, history_df, checkpoint


def plot_training_history(history_df, output_dir):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history_df["epoch"], history_df["train_loss"], label="Train MSE")
    ax.plot(history_df["epoch"], history_df["val_loss"], label="Validation MSE")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "training_curve.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


@torch.no_grad()
def save_test_predictions(model, data_loader, output_path):
    model.eval()
    rows = []
    for batch in data_loader:
        target, p_final, p_idb, p_direct, theta1, theta2, epsilon = forward_batch(
            model, batch
        )
        for i in range(target.size(0)):
            rows.append({
                "drugA_name": batch["drugA_name"][i],
                "drugB_name": batch["drugB_name"][i],
                "cell_line": batch["cell_line"][i],
                "drugA_conc": float(batch["drugA_conc"][i]),
                "drugB_conc": float(batch["drugB_conc"][i]),
                "single_resp_1": float(batch["single_resp_1"][i]),
                "single_resp_2": float(batch["single_resp_2"][i]),
                "target": float(target[i].cpu()),
                "prediction": float(p_final[i].cpu()),
                "prediction_idb": float(p_idb[i].cpu()),
                "prediction_hpb": float(p_direct[i].cpu()),
                "theta1": float(theta1[i].cpu()),
                "theta2": float(theta2[i].cpu()),
                "epsilon": float(epsilon[i].cpu()),
            })
    predictions_df = pd.DataFrame(rows)
    predictions_df.to_csv(output_path, index=False)
    return predictions_df


In [ ]:
def split_counts(train_df, val_df, test_df):
    return {
        "train_samples": len(train_df),
        "val_samples": len(val_df),
        "test_samples": len(test_df),
        "train_groups": train_df["group_id"].nunique(),
        "val_groups": val_df["group_id"].nunique(),
        "test_groups": test_df["group_id"].nunique(),
    }


def save_split_assignments(train_df, val_df, test_df, run_seed, output_dir):
    assignment_df = pd.concat(
        [
            train_df.assign(split="train"),
            val_df.assign(split="validation"),
            test_df.assign(split="test"),
        ],
        ignore_index=True,
    )
    assignment_df = assignment_df[
        ["group_id", "drugA_name", "drugB_name", "cell_line", "split"]
    ].drop_duplicates(subset=["group_id", "split"])
    assignment_df.insert(0, "seed", run_seed)
    assignment_df.to_csv(output_dir / "split_assignments.csv", index=False)


def run_single_seed(data_df, drug_features, cell_features, run_seed):
    run_dir = BASE_OUTPUT_DIR / f"seed_{run_seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = run_dir / "final_test_metrics.csv"

    if SKIP_COMPLETED and metrics_path.exists():
        print(f"Seed {run_seed}: completed result found; skipping retraining.")
        return pd.read_csv(metrics_path).iloc[0].to_dict()

    print("\n" + "=" * 78)
    print(f"Seed {run_seed}: starting a new grouped split and training from scratch")
    print("=" * 78)

    set_seed(run_seed)
    train_df, val_df, test_df = grouped_train_val_test_split(data_df, seed=run_seed)
    counts = split_counts(train_df, val_df, test_df)
    save_split_assignments(train_df, val_df, test_df, run_seed, run_dir)

    train_dataset = DrugCombinationDataset(train_df, drug_features, cell_features)
    val_dataset = DrugCombinationDataset(val_df, drug_features, cell_features)
    test_dataset = DrugCombinationDataset(test_df, drug_features, cell_features)

    loader_kwargs = {
        "batch_size": BATCH_SIZE,
        "num_workers": 0,
        "pin_memory": device.type == "cuda",
    }
    train_generator = torch.Generator().manual_seed(run_seed)
    train_loader = DataLoader(
        train_dataset, shuffle=True, generator=train_generator, **loader_kwargs
    )
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)


    model = DrugCombinationModel(
        drug_feat_dim=drug_features.shape[1],
        cell_feat_dim=cell_features.shape[1],
    )
    print(
        f"Seed {run_seed}: trainable parameters = "
        f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
    )

    model, history_df, checkpoint = train_model(
        model, train_loader, val_loader, run_seed, run_dir
    )
    plot_training_history(history_df, run_dir)

    _, test_metrics = evaluate_model(model, test_loader, include_c_index=True)
    row = {
        "seed": run_seed,
        "best_epoch": int(checkpoint["epoch"]),
        "best_validation_mse": float(checkpoint["best_val_loss"]),
        **counts,
        **test_metrics,
    }
    pd.DataFrame([row]).to_csv(metrics_path, index=False)
    save_test_predictions(model, test_loader, run_dir / "final_test_predictions.csv")
    torch.save(model.state_dict(), run_dir / "final_model_group_split_drug_fusion.pth")

    print(f"\nSeed {run_seed} independent test metrics:")
    for metric in ["mse", "rmse", "mae", "pcc", "r2", "c_index"]:
        print(f"{metric.upper()}: {row[metric]:.6f}")

    del model, train_loader, val_loader, test_loader
    del train_dataset, val_dataset, test_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row


In [ ]:
combo_df, drug_features, cell_features = load_data(DATA_DIR)
data_df = preprocess_data(combo_df, drug_features, cell_features)

all_rows = []
for run_seed in TRAIN_SEEDS:
    row = run_single_seed(
        data_df=data_df,
        drug_features=drug_features,
        cell_features=cell_features,
        run_seed=run_seed,
    )
    all_rows.append(row)

all_metrics_df = pd.DataFrame(all_rows).sort_values("seed").reset_index(drop=True)
all_metrics_df.to_csv(BASE_OUTPUT_DIR / "repeated_split_all_metrics.csv", index=False)

display_columns = [
    "seed", "best_epoch", "mse", "rmse", "mae", "pcc", "r2", "c_index",
    "train_samples", "val_samples", "test_samples",
    "train_groups", "val_groups", "test_groups",
]
display(all_metrics_df[display_columns])


In [ ]:
metric_columns = ["mse", "rmse", "mae", "pcc", "r2", "c_index"]
summary_rows = []

if len(all_metrics_df) != len(ALL_SEEDS):
    raise RuntimeError(
        f"Expected {len(ALL_SEEDS)} completed runs, but found {len(all_metrics_df)}"
    )

for metric in metric_columns:
    values = pd.to_numeric(all_metrics_df[metric], errors="raise").to_numpy(float)
    n = len(values)
    mean_value = values.mean()
    sample_sd = values.std(ddof=1)
    ci_half_width = student_t.ppf(0.975, df=n - 1) * sample_sd / np.sqrt(n)
    summary_rows.append({
        "metric": metric,
        "n": n,
        "mean": mean_value,
        "sample_sd": sample_sd,
        "minimum": values.min(),
        "maximum": values.max(),
        "ci95_lower": mean_value - ci_half_width,
        "ci95_upper": mean_value + ci_half_width,
        "mean_plus_minus_sd": f"{mean_value:.6f} ± {sample_sd:.6f}",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(BASE_OUTPUT_DIR / "repeated_split_summary.csv", index=False)
display(summary_df)

print("\nReport the five runs as mean ± sample SD:")
for _, row in summary_df.iterrows():
    print(f"{row['metric'].upper()}: {row['mean_plus_minus_sd']}")
